# BUSI70575 Final Reproducible Pipeline

This notebook is the readable entry point for the final deliverable. The modelling logic lives in `coursework/src`; the notebook explains the flow and calls the reusable source modules.

Expected outputs:

- Required metamodel probabilities: `outputs/metamodel_predictions.csv`
- Optional strategy weights: `outputs/strategy_weights.csv`
- Report PDF and packaged files: `Deliverables/`

The final coursework prediction file is frozen and hash-checked. For a genuinely new prediction period, use the final section of this notebook or `coursework/predict_new_period.py`.

## 1. Project Setup

The helper below makes the notebook work both from the original project root and from `Deliverables/code/coursework/notebooks`. It searches upward for the official input files, then imports the package code from the matching root.

In [1]:
from dataclasses import replace
from pathlib import Path
import json
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "ohlcv_data.csv").exists() and (candidate / "primary_signals.csv").exists():
            return candidate
        # When running inside Deliverables/code, the raw files may be placed next to the code folder.
        if (candidate / "code" / "ohlcv_data.csv").exists() and (candidate / "code" / "primary_signals.csv").exists():
            return candidate / "code"
    raise FileNotFoundError(
        "Could not find ohlcv_data.csv and primary_signals.csv. Place them in the project root "
        "or in Deliverables/code, then rerun this notebook."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coursework.src.config import CONFIG
from coursework.src.data_loader import load_and_merge_data
from coursework.src.pipeline import run_coursework_pipeline
from coursework.src.prediction import predict_new_period
from coursework.src.strategy import run_strategy_backtest
from coursework.src.utils import sha256_file
from coursework.src.validation import validate_input_files, validate_prediction_file

config = replace(
    CONFIG,
    ohlcv_path=PROJECT_ROOT / "ohlcv_data.csv",
    primary_signals_path=PROJECT_ROOT / "primary_signals.csv",
    output_dir=PROJECT_ROOT / "outputs",
)

{
    "project_root": str(PROJECT_ROOT),
    "ohlcv_path": str(config.ohlcv_path),
    "primary_signals_path": str(config.primary_signals_path),
    "output_dir": str(config.output_dir),
}

{'project_root': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code',
 'ohlcv_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/ohlcv_data.csv',
 'primary_signals_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/primary_signals.csv',
 'output_dir': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/outputs'}

## 2. Input Validation

The coursework inputs are the OHLCV panel and the supplied primary-signal file. This check confirms that both files exist and that the canonical date-instrument panel is usable.

In [2]:
validate_input_files(config)
raw_data = load_and_merge_data(config)
{
    "ohlcv_rows": len(raw_data["ohlcv"]),
    "signal_rows_long": len(raw_data["signals_long"]),
    "signal_dates": raw_data["signals_long"]["date"].nunique(),
    "signal_instruments": raw_data["signals_long"]["instrument"].nunique(),
}

{'ohlcv_rows': 83547,
 'signal_rows_long': 7095,
 'signal_dates': 645,
 'signal_instruments': 11}

## 3. Reproduce The Final Submission Pipeline

This calls `run_coursework_pipeline(config)`. Internally it loads data, builds lagged features, constructs triple-barrier meta-labels, loads the frozen model-suite evidence, validates the promoted final predictions, computes/loads evaluation evidence, and writes final audit files.

Important: this step validates the frozen public-2022H1 submission file instead of overwriting it.

In [3]:
artifacts = run_coursework_pipeline(config)
{
    "prediction_path": str(artifacts.prediction_path),
    "audit_path": str(artifacts.audit_path),
    "final_model": artifacts.final_model,
    "prediction_rows": len(artifacts.final_predictions),
}

{'prediction_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/outputs/metamodel_predictions.csv',
 'audit_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/outputs/final_integration_audit.txt',
 'final_model': {'name': 'Calibrated 0.50 Logistic + 0.50 signal-history MLP blend',
  'type': 'frozen_probability_blend',
  'hmm_enabled': False,
  'prediction_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/outputs/metamodel_predictions.csv',
  'mean_roc_auc': 0.583,
  'mean_f1': 0.626},
 'prediction_rows': 1408}

## 4. Required Prediction File Checks

The required output must have exact columns `date,instrument,prediction`, no duplicate date-instrument rows, no missing values, and probabilities in `[0, 1]`.

In [4]:
prediction_summary = validate_prediction_file(config.final_prediction_path)
prediction_hash = sha256_file(config.final_prediction_path)
{
    **prediction_summary,
    "sha256": prediction_hash,
    "hash_matches_frozen_final": prediction_hash == config.promoted_final_hash,
}

{'rows': 1408,
 'start_date': Timestamp('2022-01-03 00:00:00'),
 'end_date': Timestamp('2022-06-30 00:00:00'),
 'n_dates': 128,
 'n_instruments': 11,
 'instruments': ['cl1s',
  'es1s',
  'fesx1s',
  'gc1s',
  'hg1s',
  'ho1s',
  'ng1s',
  'nq1s',
  'pl1s',
  'rb1s',
  'si1s'],
 'sha256': 'c5c7ca869d905b384ef3c9072c3377e0f43c7a7ad03c9125aa062077f0f9b369',
 'hash_matches_frozen_final': True}

## 5. Optional Strategy Output

The optional strategy converts metamodel probabilities and primary-signal direction into signed portfolio weights. It does not change the required prediction file.

In [5]:
strategy_result = run_strategy_backtest(config)
{
    "selected_method": strategy_result["selected_method"],
    "strategy_weights_path": strategy_result["strategy_weights_path"],
    "metrics_path": strategy_result["metrics_path"],
    "validation": strategy_result["validation"],
    "final_prediction_hash": strategy_result["final_prediction_hash"],
}

{'selected_method': 'soft_allocation',
 'strategy_weights_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/outputs/strategy_weights.csv',
 'metrics_path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Deliverables/code/outputs/strategy_backtest_metrics.csv',
 'validation': {'rows': 1408,
  'date_count': 128,
  'instrument_count': 11,
  'min_weight': -0.25,
  'max_weight': 0.1666665,
  'max_abs_weight': 0.25,
  'gross_exposure_mean': 0.97656153125,
  'gross_exposure_max': 0.9999990000000001,
  'zero_signal_nonzero_weight': 0,
  'direction_misaligned_weight': 0},
 'final_prediction_hash': 'c5c7ca869d905b384ef3c9072c3377e0f43c7a7ad03c9125aa062077f0f9b369'}

## 6. Deliverables Sanity Check

This checks the files that should exist after running `python3 build_deliverables.py` from the project root. The raw data are not duplicated in `Deliverables/`; they are official coursework inputs.

In [6]:
deliverables_dir = PROJECT_ROOT / "Deliverables"
if not deliverables_dir.exists() and PROJECT_ROOT.name == "code" and PROJECT_ROOT.parent.name == "Deliverables":
    deliverables_dir = PROJECT_ROOT.parent
expected_files = [
    deliverables_dir / "report" / "BUSI70575_Report.pdf",
    deliverables_dir / "required" / "metamodel_predictions.csv",
    deliverables_dir / "optional_strategy" / "strategy_weights.csv",
    deliverables_dir / "code" / "coursework" / "notebooks" / "00_final_reproducible_pipeline.ipynb",
    deliverables_dir / "sha256_manifest.txt",
]
def display_path(path: Path) -> str:
    for base in [PROJECT_ROOT, deliverables_dir]:
        try:
            return str(path.relative_to(base))
        except ValueError:
            pass
    return str(path)

deliverable_check = {display_path(path): path.exists() for path in expected_files}
deliverable_check

{'report/BUSI70575_Report.pdf': True,
 'required/metamodel_predictions.csv': True,
 'optional_strategy/strategy_weights.csv': True,
 'coursework/notebooks/00_final_reproducible_pipeline.ipynb': True,
 'sha256_manifest.txt': True}

## 7. Predicting A New Period

For a new window, provide an extended OHLCV file and an extended primary-signal file. The files must include enough history before the prediction start because the model uses lagged features and trains only on data strictly before the requested prediction window.

The cell below is off by default. Change `RUN_NEW_PERIOD_EXAMPLE` to `True` and update the paths/dates when new data are available.

In [7]:
RUN_NEW_PERIOD_EXAMPLE = False

if RUN_NEW_PERIOD_EXAMPLE:
    new_config = replace(
        CONFIG,
        ohlcv_path=PROJECT_ROOT / "ohlcv_data_extended.csv",
        primary_signals_path=PROJECT_ROOT / "primary_signals_extended.csv",
        prediction_start="2022-07-01",
        prediction_end="2022-12-31",
        output_dir=PROJECT_ROOT / "outputs",
    )
    new_result = predict_new_period(
        new_config,
        output_path=PROJECT_ROOT / "outputs" / "new_period_predictions.csv",
    )
    display(new_result)
else:
    print("New-period prediction example skipped. Set RUN_NEW_PERIOD_EXAMPLE = True when extended data are available.")

New-period prediction example skipped. Set RUN_NEW_PERIOD_EXAMPLE = True when extended data are available.
